In [ ]:
import numpy as np                     # 引入基础软件包numpy
import pandas as pd                    # 引入基础软件包pandas
import statsmodels.api as sm  # 引入Logistic regression软件包statsmodels
from sklearn.model_selection import train_test_split # 引入训练集/测试集构造工具包
from sklearn import metrics            # 引入模型评价指标AUC计算工具包
import matplotlib.pyplot as plt        # 引入绘图软件包
import scipy     

In [ ]:
# 数据读入
data_path =  u'./Data/wide_data.csv'
raw_data = pd.read_table(data_path, sep=',', header=0)

In [ ]:
raw_data.head(5).T

In [ ]:
raw_data['y'] = raw_data['MODE'].apply(lambda x : 1 if x==3 else 0)

In [ ]:
model_data = raw_data[[
    'OBS_ID'
    , 'HINC', 'PSIZE'
    , 'TTME_AIR', 'TTME_TRAIN', 'TTME_BUS'
    , 'INVC_AIR', 'INVC_TRAIN', 'INVC_BUS', 'INVC_CAR'
    , 'INVT_AIR', 'INVT_TRAIN','INVT_BUS', 'INVT_CAR'
    , 'y'
]]

In [ ]:
model_data.describe().T

In [ ]:
model_data.info()                      # 查看每一列的数据类型 & 数值缺失情况
# 重要返回信息
# | RangeIndex: 210 entries, 0 to 209
# | Data columns (total 9 columns):
# | ...
# | HINC              210 non-null int64
# | ...
# 1.缺失值处理
model_data = model_data.dropna()       # 缺失值处理，删除
model_data = model_data.fillna(0)      # 缺失值处理，填充（零，均值，中位数，预测值等）

In [ ]:
# 2.数值型
import re                               # 正则表达式工具包
float_patten = '^(-?\\d+)(\\.\\d+)?$'   # 定义浮点数正则patten
float_re = re.compile(float_patten)     # 编译
model_data['HINC'][model_data['HINC'].apply(lambda x : 'not_float' if float_re.match(str(x)) == None else 'float') == 'not_float'] # 查看非浮点型数据
# 重要返回信息
# | 2    null
# | Name: distance, dtype: object

In [ ]:
# 离散变量分析
multi_categorical_x_list = ['PSIZE']

print('======离散变量显著性======')
for categorical_x in multi_categorical_x_list:
    crosstab = pd.crosstab( model_data['y'],model_data[categorical_x])
    p=scipy.stats.chi2_contingency(crosstab)[1]
    print(categorical_x,":",p)

In [ ]:
# 连续变量分析
continuous_x_list = ['HINC','TTME_AIR', 'TTME_TRAIN', 'TTME_BUS'
    ,'INVC_AIR', 'INVC_TRAIN', 'INVC_BUS', 'INVC_CAR'
    , 'INVT_AIR', 'INVT_TRAIN','INVT_BUS', 'INVT_CAR' ]
result = []
for continuous_x in continuous_x_list:
    logistic = sm.Logit(model_data['y'],model_data[continuous_x]).fit()
    p = logistic.pvalues[continuous_x]
    y_predict = logistic.predict(model_data[continuous_x])
    AUC = metrics.roc_auc_score(model_data['y'],y_predict)
    result.append(continuous_x+":"+str(p)+'  AUC:'+str(AUC))
print('======连续变量显著性======')
for i in result:
    print(i)

In [ ]:
crosstab = pd.crosstab( model_data['y'],model_data['PSIZE'])
p=scipy.stats.chi2_contingency(crosstab)[1]
print("PSIZE:",p)

In [ ]:
logistic = sm.Logit(model_data['y'],model_data['INVT_CAR']).fit()
p = logistic.pvalues['INVT_CAR']
y_predict = logistic.predict(model_data['INVT_CAR'])
AUC = metrics.roc_auc_score(model_data['y'],y_predict)
result = 'INVT_CAR:'+str(p)+'  AUC:'+str(AUC)
print(result)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor #诊断多重共线性，vif
X = model_data[[
     'HINC', 'PSIZE'
    ,'TTME_AIR', 'TTME_TRAIN', 'TTME_BUS'
    ,'INVC_AIR', 'INVC_TRAIN', 'INVC_BUS', 'INVC_CAR'
    , 'INVT_AIR', 'INVT_TRAIN','INVT_BUS', 'INVT_CAR'
]]

vif = pd.DataFrame()
vif['VIF Factor'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif['features'] = X.columns
print('================多重共线性==============')
print(vif)

In [ ]:
# 建模数据构造
X = model_data[[ 'HINC','PSIZE','TTME_TRAIN' , 'INVC_CAR']]
y = raw_data['y']
# 哑变量处理
dummies = pd.get_dummies(X['PSIZE'], drop_first=False)
dummies.columns = [ 'PSIZE'+'_'+str(x) for x in dummies.columns.values]
X = pd.concat([X, dummies], axis=1)
X = X.drop('PSIZE',axis=1) # 删去原离散变量
# X = X.drop('PSIZE_1',axis=1) 
# X = X.drop('PSIZE_2',axis=1) 
# X = X.drop('PSIZE_3',axis=1) 
X = X.drop('PSIZE_4',axis=1) 
X = X.drop('PSIZE_5',axis=1) 
X = X.drop('PSIZE_6',axis=1) 

X['Intercept'] = 1

# 训练集与测试集的比例为80%和20%
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.8, random_state=1234)

# 全变量建模
logistic = sm.Logit(y_train,X_train).fit()
print(logistic.summary2())
# 重要返回信息
# | ------------------------------------------------------------------
# |                Coef.   Std.Err.     z     P>|z|    [0.025   0.975]
# | ------------------------------------------------------------------
# | HINC           0.0264    0.0100   2.6477  0.0081   0.0068   0.0459
# | TTME_TRAIN     0.0389    0.0195   1.9916  0.0464   0.0006   0.0772
# | INVC_CAR      -0.0512    0.0204  -2.5103  0.0121  -0.0913  -0.0112
# | PSIZE_1       -0.3077    0.7317  -0.4206  0.6741  -1.7419   1.1264
# | PSIZE_2       -1.0800    0.6417  -1.6829  0.0924  -2.3378   0.1778
# | PSIZE_3       -0.7585    0.7582  -1.0004  0.3171  -2.2444   0.7275
# | Intercept     -1.8879    1.1138  -1.6951  0.0901  -4.0708   0.2950
# | =================================================================
print("========训练集AUC========")
y_train_predict = logistic.predict(X_train)
print(metrics.roc_auc_score(y_train,y_train_predict))
print("========测试集AUC========")
y_test_predict = logistic.predict(X_test)
print(metrics.roc_auc_score(y_test,y_test_predict))